In [0]:
# Databricks notebook source
# ETL - Squad 3 - ecommerce_rastreamento_entregas
# Fluxo: Raw CSV -> Bronze Delta -> Silver Delta -> Gold Delta/SQL Server

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
SOURCE_FILE = "ecommerce_rastreamento_entregas.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false",
}

EXPECTED_COLUMNS = [
    "id_rastreamento",
    "id_pedido_ecommerce",
    "codigo_rastreio",
    "id_transportadora",
    "status_entrega",
    "dt_evento",
    "observacao",
]

BRONZE_TABLE = "ecommerce_rastreamento_entregas"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

PARTITION_DATE_COLUMN = "dt_evento"
BRONZE_WRITE_MODE = "append"

print("SOURCE_PATH:", SOURCE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("PARTITION_DATE_COLUMN:", PARTITION_DATE_COLUMN)

In [0]:
adls_options = get_adls_options()

print("Opções ADLS configuradas.")

In [0]:
# ler CSV da Raw

df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

# validar colunas esperadas
actual_columns = df_source.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

if missing_columns:
    raise Exception(f"Colunas obrigatórias ausentes na origem: {missing_columns}")

if extra_columns:
    print(f"Atenção: existem colunas extras na origem: {extra_columns}")
else:
    print("Validação OK: todas as colunas esperadas foram encontradas.")

df_source.printSchema()
display(df_source.limit(10))

In [0]:
# contar origem

total_source = df_source.count()

print(f"Total de registros lidos da Raw: {total_source}")

In [0]:
# validar conversão da data de particionamento

from pyspark.sql.functions import col, to_timestamp, count, when

df_test_date = df_source.withColumn(
    "dt_evento_convertida",
    to_timestamp(col(PARTITION_DATE_COLUMN))
)

df_validacao_data = df_test_date.select(
    count("*").alias("total_linhas"),
    count(when(col(PARTITION_DATE_COLUMN).isNull(), True)).alias("dt_evento_nula_origem"),
    count(
        when(
            col(PARTITION_DATE_COLUMN).isNotNull() &
            col("dt_evento_convertida").isNull(),
            True
        )
    ).alias("falhas_conversao")
)

display(df_validacao_data)

validacao_data = df_validacao_data.collect()[0]

if validacao_data["falhas_conversao"] > 0:
    raise Exception("Existem valores de dt_evento que não foram convertidos para timestamp.")

print("Validação OK: dt_evento pode ser usada para particionamento.")

In [0]:
# criar DataFrame Bronze

from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_timestamp,
    year,
    month,
    sha2,
    concat_ws,
    coalesce,
    lit
)

hash_columns = [
    coalesce(col(c).cast("string"), lit("__NULL__"))
    for c in EXPECTED_COLUMNS
]

df_bronze = (
    df_source
    .select(
        *[col(c).cast("string").alias(c) for c in EXPECTED_COLUMNS],
        col("_metadata.file_path").alias("bronze_source_file")
    )
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_record_hash", sha2(concat_ws("||", *hash_columns), 256))
    .withColumn("ano", year(to_timestamp(col(PARTITION_DATE_COLUMN))))
    .withColumn("mes", month(to_timestamp(col(PARTITION_DATE_COLUMN))))
)

df_bronze.printSchema()
display(df_bronze.limit(10))

In [0]:
# validar partições criadas

display(
    df_bronze
    .groupBy("ano", "mes")
    .count()
    .orderBy("ano", "mes")
)

In [0]:
# validar se existem registros sem partição

from pyspark.sql.functions import count, when, col

df_validacao_particoes = df_bronze.select(
    count("*").alias("total_linhas"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo")
)

display(df_validacao_particoes)

validacao_particoes = df_validacao_particoes.collect()[0]

if validacao_particoes["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Bronze.")

if validacao_particoes["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Bronze.")

print("Validação OK: partições ano e mes criadas corretamente.")

In [0]:
# filtrar apenas registros ainda não carregados na Bronze

DEDUP_COLUMNS = ["bronze_source_file", "bronze_record_hash"]

df_bronze_batch = df_bronze.dropDuplicates(DEDUP_COLUMNS)

try:
    df_bronze_existing = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(BRONZE_PATH)
    )

    total_bronze_before = df_bronze_existing.count()

    if "bronze_record_hash" in df_bronze_existing.columns:
        df_existing_keys = (
            df_bronze_existing
            .select(*DEDUP_COLUMNS)
            .distinct()
        )
    else:
        existing_hash_columns = [
            coalesce(col(c).cast("string"), lit("__NULL__"))
            for c in EXPECTED_COLUMNS
        ]

        df_existing_keys = (
            df_bronze_existing
            .withColumn(
                "bronze_record_hash",
                sha2(concat_ws("||", *existing_hash_columns), 256)
            )
            .select(*DEDUP_COLUMNS)
            .distinct()
        )

    df_bronze_to_write = (
        df_bronze_batch
        .join(df_existing_keys, DEDUP_COLUMNS, "left_anti")
    )

    print("Bronze existente encontrada.")

except Exception:
    total_bronze_before = 0
    df_bronze_to_write = df_bronze_batch

    print("Bronze ainda não existe. Primeira carga incremental.")

total_batch = df_bronze_batch.count()
total_to_write = df_bronze_to_write.count()

print(f"Total lido nesta execução: {total_source}")
print(f"Total após deduplicar o próprio lote: {total_batch}")
print(f"Total novo a gravar: {total_to_write}")

In [0]:
# gravar Bronze Delta

if total_to_write > 0:
    (
        df_bronze_to_write
        .write
        .format("delta")
        .mode(BRONZE_WRITE_MODE)
        .option("mergeSchema", "true")
        .options(**adls_options)
        .partitionBy("ano", "mes")
        .save(BRONZE_PATH)
    )

    print(f"Novos registros gravados com sucesso na Bronze: {BRONZE_PATH}")

else:
    print("Nenhum registro novo para gravar. Bronze não foi alterada.")

In [0]:
# validar Bronze incremental

df_bronze_delta = (
    spark
    .read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

total_bronze_after = df_bronze_delta.count()

print(f"Total Bronze antes: {total_bronze_before}")
print(f"Total novo gravado nesta execução: {total_to_write}")
print(f"Total Bronze depois: {total_bronze_after}")

if total_bronze_after != total_bronze_before + total_to_write:
    raise Exception("Erro: total da Bronze depois não bate com o esperado.")

display(
    df_bronze_delta
    .groupBy("ano", "mes")
    .count()
    .orderBy("ano", "mes")
)

df_validacao_bronze_final = df_bronze_delta.select(
    count("*").alias("total_linhas"),
    count(when(col("ano").isNull(), True)).alias("ano_nulo"),
    count(when(col("mes").isNull(), True)).alias("mes_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo")
)

display(df_validacao_bronze_final)

validacao_bronze_final = df_validacao_bronze_final.collect()[0]

if validacao_bronze_final["ano_nulo"] > 0:
    raise Exception("Erro: existem registros com ano nulo na Bronze.")

if validacao_bronze_final["mes_nulo"] > 0:
    raise Exception("Erro: existem registros com mes nulo na Bronze.")

if validacao_bronze_final["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao_bronze_final["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

print("Validação incremental da Bronze OK.")